# 소득세법 RAG 데모 — "내 월급, 실수령액은 얼마?"

이 노트북은 **RAG(Retrieval-Augmented Generation)** 구조를 실제 법령 데이터로 시연합니다.

- 사용자가 **월급(세전)** 을 입력하면
- **DOCX 파일 하나**(`data/소득세법_통합.docx`)를 직접 열어 문단·표를 원문 순서 그대로
  읽어(python-docx) 관련 조문을 검색(Retrieval)하고
- 세금·보험료를 **직접 계산**한 뒤
- 검색된 조문을 근거로 **자연어 설명**을 생성(Generation)합니다

## 왜 이 구조인가?

세금 계산 자체는 규칙 기반(rule-based)이라 LLM이 계산을 담당하지 않습니다.
LLM은 오직 **"계산 결과 + 검색된 법조문을 자연스러운 설명으로 풀어쓰는"** 역할만 맡습니다.
→ 계산은 100% 정확(환각 없음), LLM은 설명/근거 인용에만 사용 → 토큰 비용을 최소화합니다.

## 데이터: 하나의 DOCX로 정리된 원본

`data/소득세법_통합.docx` 한 파일에 **소득세법 259개 조문(삭제조문 제외) + 4대보험·지방세법
5개 조문**이 원문 순서 그대로 들어 있습니다. 이 파일은:

1. 법제처 국가법령정보센터의 실제 소득세법 조문 데이터(JSONL, 349건)와 공식 PDF 원문을
   기반으로 하고,
2. 그중 원문 PDF에서 표가 이미지로만 렌더링되어 자동 텍스트 추출에 실패했던 조문
   (근로소득공제 제47조, 종합소득세율 제55조, 근로소득세액공제 제59조)은 그 표 이미지를
   **직접 육안으로 대조**해 실제 워드 **표(Table)** 로 원래 있어야 할 위치에 복원해 넣었고,
3. 그 검증 과정에서 근로소득세액공제 한도 계산식의 실제 비율이 5%가 아니라 **1/2(50%)**
   라는 것도 발견해 바로잡았습니다.

(`preprocessing.ipynb`가 이 DOCX를 만드는 전처리 노트북입니다 — 재현하려면 그 노트북을
먼저 실행하세요. 이 노트북은 이미 만들어진 DOCX를 읽기만 합니다.)

**이 노트북의 RAG 인제스천은 이 DOCX 파일 하나를 `python-docx`로 직접 파싱하는 것으로
시작합니다** — 미리 손질된 JSON을 읽는 게 아니라, 실제 문서 구조(제목 스타일·문단·표)를
순서대로 순회하며 조문 단위 청크를 만듭니다.

## 비용 구조 (LLM에 돈 안 쓰는 법)

| 구성요소 | 사용 기술 | 비용 |
|---|---|---|
| DOCX 파싱 | `python-docx` (로컬 실행) | **무료** |
| 임베딩(검색용 벡터화) | `sentence-transformers` (한국어 특화 모델, 로컬 실행) | **무료** (로컬 CPU) |
| 벡터 검색 | numpy 코사인 유사도 + 키워드 하이브리드 | **무료** |
| 세금 계산 | 순수 파이썬 함수 (검증된 수치를 코드화) | **무료** |
| 설명 생성 | [Google AI Studio](https://aistudio.google.com/apikey) Gemini 무료 티어 API | **무료** (가입만 하면 무료 API 키 발급, 분당/일 요청 제한 있음) |

> 원래는 Groq 무료 API를 쓰려 했으나, 일부 네트워크(회사망·특정 ISP 등)에서는 Groq API
> 자체가 접속 차단되어 있을 수 있습니다(`Access denied. Please check your network settings.`).
> 이 경우 API 키와 무관하게 요청이 막히므로, 접속이 확인된 Google Gemini 무료 티어를
> 사용합니다. Gemini API 키가 없어도 노트북은 끝까지 실행됩니다 — 마지막 "설명 생성" 셀만
> 자동으로 "LLM 없이 계산 결과 + 근거 조문만 출력"하는 폴백 모드로 동작합니다.
>
> **모델 이름은 수시로 바뀝니다.** 예: `gemini-2.5-flash`는 2026-07 기준 신규 사용자에게
> 차단되어 `gemini-3.5-flash-lite`로 교체했습니다. `NOT_FOUND` 오류가 나면
> [Google AI Studio](https://aistudio.google.com/) 또는 `client.models.list()`로 현재
> 사용 가능한 모델명을 확인해 아래 `GEMINI_MODEL` 값을 바꿔주세요.

## 가상환경: uv

```bash
uv venv                       # .venv 생성
uv pip install -r requirements.txt
source .venv/bin/activate
jupyter notebook income_tax_rag_demo.ipynb
```

## ⚠️ 면책 (Disclaimer)

이 노트북의 세율·요율·공제 수치는 **교육/시연 목적**입니다.
- 매년 바뀌는 4대보험 요율은 최신 고시와 다를 수 있습니다.
- 실제 원천징수는 국세청 **근로소득 간이세액표**를 따르며, 연말정산으로 최종 정산됩니다.
- **실제 세무 신고·상담은 반드시 국세청 홈택스 또는 세무 전문가를 통해 확인하세요.**


## 1. 환경 준비

필요한 패키지: `python-docx`(DOCX 파싱), `sentence-transformers`(임베딩), `numpy`(유사도 계산), `google-genai`(무료 Gemini API 클라이언트), `python-dotenv`(API 키 로드)

In [1]:
# uv로 이미 설치했다면 아래는 생략 가능합니다.
# uv pip install -r requirements.txt

import re
import os
import json
from pathlib import Path

import numpy as np

DOCX_PATH = Path("data/소득세법_통합.docx")
print("RAG 원본 DOCX:", DOCX_PATH.resolve())


RAG 원본 DOCX: /home/dkbmc-lab/Desktop/RAG/data/소득세법_통합.docx


## 2. DOCX 인제스천 (Ingestion) — 문서를 원문 순서 그대로 읽기

`python-docx`의 `iter_inner_content()`는 문단(Paragraph)과 표(Table)를 **문서에 실제로
등장하는 순서 그대로** 순회할 수 있게 해줍니다. 이를 이용해:

- `Heading 3` 스타일 문단("제47조(근로소득공제)" 같은)을 만나면 새 조문 시작
- `Heading 1` 스타일 문단을 만나면 "4대보험 보충" 섹션으로 전환
- 장/절 표시 문단("제1장 총칙" 등)은 본문이 아니라 위계 메타데이터로 기록
- **표를 만나면** 헤더 행 기준으로 각 행을 "구간: … / 공제액: …" 형태 텍스트로 풀어서
  (linearize) 해당 조문의 본문에 이어붙임

이렇게 조문 단위로 재구성합니다.


In [2]:
import docx
from docx.text.paragraph import Paragraph
from docx.table import Table

CHAPTER_RE = re.compile(r"^제\d+(장|절|관)\b")
SOCIAL_INS_HEADING_RE = re.compile(r"^(?P<law>.+?)\s+(?P<article>제[^(]+)\((?P<topic>[^)]*)\)$")
INCOME_TAX_HEADING_RE = re.compile(r"^(?P<article>제[^(]+)\((?P<topic>[^)]*)\)$")


def table_to_text(table: Table) -> str:
    """표의 각 행을 '헤더1: 값1 / 헤더2: 값2' 형태 텍스트로 풀어쓴다 (표 linearization)."""
    rows = [[c.text.strip() for c in row.cells] for row in table.rows]
    header, body_rows = rows[0], rows[1:]
    lines = []
    for row in body_rows:
        lines.append(" / ".join(f"{h}: {v}" for h, v in zip(header, row)))
    return "\n".join(lines)


def parse_docx(path: Path) -> list[dict]:
    document = docx.Document(path)
    articles: list[dict] = []
    current: dict | None = None
    current_위계 = None
    in_social_section = False

    def flush():
        nonlocal current
        if current is not None and current["text"].strip():
            articles.append(current)
        current = None

    for item in document.iter_inner_content():
        if isinstance(item, Paragraph):
            style = item.style.name if item.style else ""
            text = item.text.strip()
            if not text:
                continue

            if style == "Heading 1":
                flush()
                in_social_section = True
                current_위계 = None
                continue
            if style == "Title":
                continue  # 문서 대제목("소득세법")은 조문이 아님

            if style == "Heading 3":
                flush()
                if in_social_section:
                    m = SOCIAL_INS_HEADING_RE.match(text)
                    law, article, topic = (m.group("law"), m.group("article"), m.group("topic")) if m else ("", text, "")
                else:
                    m = INCOME_TAX_HEADING_RE.match(text)
                    law, article, topic = ("소득세법", m.group("article"), m.group("topic")) if m else ("소득세법", text, "")
                current = {"law": law, "article": article, "topic": topic, "위계": current_위계, "text": ""}
                continue

            if CHAPTER_RE.match(text) and not in_social_section:
                current_위계 = text
                continue

            if current is not None:
                current["text"] += text + "\n"

        elif isinstance(item, Table):
            if current is not None:
                current["text"] += table_to_text(item) + "\n"

    flush()
    for a in articles:
        a["text"] = a["text"].strip()
    return articles


active_articles = parse_docx(DOCX_PATH)
print(f"DOCX에서 파싱한 조문 수: {len(active_articles)}")
for a in active_articles[:5]:
    print(f"  [{a['law']} {a['article']}] {a['topic']}  ({len(a['text'])}자)")
print("  ...")


DOCX에서 파싱한 조문 수: 268
  [소득세법 제1조] 목적  (117자)
  [소득세법 제1조의2] 정의  (280자)
  [소득세법 제2조] 납세의무  (1208자)
  [소득세법 제2조의2] 납세의무의 범위  (562자)
  [소득세법 제2조의3] 신탁재산 귀속 소득에 대한 납세의무의 범위  (193자)
  ...


## 3. 청킹 (Chunking)

법 조문은 길이 편차가 큽니다. 너무 긴 조문을 통째로 임베딩하면 검색 정확도가 떨어지므로,
항(①②③...) 단위로 쪼개고 너무 잘게 쪼개지지 않도록 적당한 크기로 다시 묶습니다.


In [3]:
CIRCLED_NUMS = "①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮⑯⑰⑱⑲⑳"
CHUNK_MAX_CHARS = 900


def split_into_chunks(text: str) -> list[str]:
    if len(text) <= CHUNK_MAX_CHARS:
        return [text]

    parts = [p for p in re.split(f"(?=[{CIRCLED_NUMS}])", text) if p.strip()]
    if len(parts) <= 1:
        return [text[i:i + CHUNK_MAX_CHARS] for i in range(0, len(text), CHUNK_MAX_CHARS)]

    chunks, buf = [], ""
    for p in parts:
        if len(buf) + len(p) <= CHUNK_MAX_CHARS:
            buf += p
        else:
            if buf:
                chunks.append(buf)
            buf = p
    if buf:
        chunks.append(buf)
    return chunks


def build_corpus(articles: list[dict]) -> list[dict]:
    corpus = []
    for idx, a in enumerate(articles):
        chunks = split_into_chunks(a["text"])
        for i, chunk_text in enumerate(chunks):
            corpus.append({
                "id": f"art{idx:03d}_{i}" if len(chunks) > 1 else f"art{idx:03d}",
                "law": a["law"],
                "article": a["article"],
                "topic": a["topic"],
                "text": chunk_text,
                "위계": a["위계"],
                "chunk_info": f"{i + 1}/{len(chunks)}" if len(chunks) > 1 else None,
            })
    return corpus


corpus = build_corpus(active_articles)
print(f"통합 코퍼스: 총 {len(corpus)}개 청크 (조문 {len(active_articles)}건에서 생성)")


통합 코퍼스: 총 357개 청크 (조문 268건에서 생성)


## 4. 임베딩 기반 하이브리드 검색 (Retrieval)

한국어 문장 의미 유사도(STS)에 특화된 임베딩 모델(`ko-sroberta-multitask`)로 코퍼스를
벡터화하고, 질문이 들어오면 코사인 유사도로 가장 관련 있는 청크를 찾습니다. 모델은 최초
1회 다운로드된 뒤 로컬 캐시에서 재사용되므로 이후 실행은 인터넷 연결 없이도 동작합니다.

> **참고 — 순수 임베딩만으로는 한계가 있습니다.** "근로소득공제"와 "근로소득세액공제"처럼
> 표현이 비슷한 두 조문은 코사인 유사도만으로는 종종 헷갈립니다. 아래 `retrieve()` 함수는
> 질문어가 청크의 주제(topic, 즉 조제목)와 정확히 일치할 때만 강하게 부스트하는
> **하이브리드(임베딩+키워드) 검색**을 적용해 이 문제를 보완합니다 — 실무 RAG에서 흔히
> 쓰는 dense+sparse 결합의 축소판입니다.


In [4]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "jhgan/ko-sroberta-multitask"

print("임베딩 모델 로드 중... (최초 1회는 다운로드 시간이 걸릴 수 있습니다)")
embedder = SentenceTransformer(EMBED_MODEL_NAME)

corpus_texts = [f"{doc['topic']}: {doc['text']}" for doc in corpus]
corpus_embeddings = embedder.encode(corpus_texts, normalize_embeddings=True, show_progress_bar=True)

print("코퍼스 임베딩 완료:", corpus_embeddings.shape)


임베딩 모델 로드 중... (최초 1회는 다운로드 시간이 걸릴 수 있습니다)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

코퍼스 임베딩 완료: (357, 768)


In [5]:
def retrieve(query: str, top_k: int = 3, keyword_boost: float = 0.5):
    """질문(query)과 가장 유사한 청크 top_k개를 반환 (임베딩 + 키워드 하이브리드).

    질의와 조제목이 '정확히 일치'할 때만 강한 부스트를 주고, 조제목이 더 긴 자유 서술형
    질의 안에 '완전한 문구로' 포함된 경우에만 약한 부스트를 줍니다 — 반대 방향(질의가
    조제목의 부분 문자열인 경우, 예: '세율' in '양도소득세의 세율')은 부스트하지 않습니다.
    """
    query_emb = embedder.encode([query], normalize_embeddings=True)[0]
    scores = corpus_embeddings @ query_emb  # normalize_embeddings=True 이므로 내적 = 코사인 유사도
    for i, doc in enumerate(corpus):
        if doc["topic"] == query:
            scores[i] += keyword_boost
        elif doc["topic"] and doc["topic"] in query:
            scores[i] += keyword_boost * 0.6
    top_idx = np.argsort(-scores)[:top_k]
    return [(corpus[i], float(scores[i])) for i in top_idx]


# 자유 서술형 질문으로 검색 기능 시연
demo_query = "월급 500만원인데 세금이랑 4대보험 얼마나 떼가?"
print(f"질문: {demo_query}\n")
for doc, score in retrieve(demo_query, top_k=5):
    print(f"[유사도 {score:.3f}] {doc['law']} {doc['article']} — {doc['topic']}")


질문: 월급 500만원인데 세금이랑 4대보험 얼마나 떼가?

[유사도 0.595] 소득세법 제47조 — 근로소득공제
[유사도 0.577] 소득세법 제12조 — 비과세소득
[유사도 0.560] 소득세법 제143조의4 — 공적연금소득세액의 연말정산
[유사도 0.555] 소득세법 제59조의3 — 연금계좌세액공제
[유사도 0.554] 소득세법 제137조 — 근로소득세액의 연말정산


## 5. 세금·4대보험료 계산 (Rule-based, LLM 미사용)

법조문의 수치를 코드화합니다. 특히 근로소득세액공제 한도(제59조 ②항 3·4호)는 초과분에
곱하는 비율이 `1/2`(50%)임을 원문 표 이미지 직접 대조 + 웹 검색 교차 확인으로 검증했습니다
— 얼핏 `5%`의 오타처럼 보이지만 실제 법령이 그렇습니다.


In [6]:
def calc_근로소득공제(총급여: float) -> float:
    """소득세법 제47조"""
    if 총급여 <= 5_000_000:
        공제 = 총급여 * 0.70
    elif 총급여 <= 15_000_000:
        공제 = 3_500_000 + (총급여 - 5_000_000) * 0.40
    elif 총급여 <= 45_000_000:
        공제 = 7_500_000 + (총급여 - 15_000_000) * 0.15
    elif 총급여 <= 100_000_000:
        공제 = 12_000_000 + (총급여 - 45_000_000) * 0.05
    else:
        공제 = 14_750_000 + (총급여 - 100_000_000) * 0.02
    return min(공제, 20_000_000)


# 종합소득세율표: (과세표준 상한, 세율, 누진공제) — 소득세법 제55조
TAX_BRACKETS = [
    (14_000_000, 0.06, 0),
    (50_000_000, 0.15, 1_260_000),
    (88_000_000, 0.24, 5_760_000),
    (150_000_000, 0.35, 15_440_000),
    (300_000_000, 0.38, 19_940_000),
    (500_000_000, 0.40, 25_940_000),
    (1_000_000_000, 0.42, 35_940_000),
    (float("inf"), 0.45, 65_940_000),
]


def calc_산출세액(과세표준: float) -> float:
    """소득세법 제55조: 과세표준 x 세율 - 누진공제"""
    if 과세표준 <= 0:
        return 0.0
    for 상한, 세율, 누진공제 in TAX_BRACKETS:
        if 과세표준 <= 상한:
            return max(과세표준 * 세율 - 누진공제, 0.0)
    return 0.0


def calc_근로소득세액공제(산출세액: float, 총급여: float) -> float:
    """소득세법 제59조. ①항은 55%/30%, ②항 한도는 7천만원 초과부터 초과분의 1/2씩 감소."""
    if 산출세액 <= 1_300_000:
        공제 = 산출세액 * 0.55
    else:
        공제 = 715_000 + (산출세액 - 1_300_000) * 0.30

    if 총급여 <= 33_000_000:
        한도 = 740_000
    elif 총급여 <= 70_000_000:
        한도 = max(740_000 - (총급여 - 33_000_000) * 0.008, 660_000)
    elif 총급여 <= 120_000_000:
        한도 = max(660_000 - (총급여 - 70_000_000) * 0.5, 500_000)
    else:
        한도 = max(500_000 - (총급여 - 120_000_000) * 0.5, 200_000)

    return min(공제, 한도)


def calc_4대보험료(월급여: float) -> dict:
    """국민연금법 제88조 / 국민건강보험법 제73조 / 노인장기요양보험법 제9조 / 고용보험법 제13조"""
    국민연금_상한 = 5_900_000  # 2024년 기준 기준소득월액 상한(근사치, 매년 변경)
    연금_기준소득 = min(월급여, 국민연금_상한)

    국민연금 = 연금_기준소득 * 0.045
    건강보험 = 월급여 * 0.03545
    장기요양보험 = 건강보험 * 0.1295
    고용보험 = 월급여 * 0.009

    return {
        "국민연금": round(국민연금),
        "건강보험": round(건강보험),
        "장기요양보험": round(장기요양보험),
        "고용보험": round(고용보험),
    }


print("계산 함수 정의 완료")


계산 함수 정의 완료


In [7]:
def calculate_take_home_pay(월급여: float, 인적공제수: int = 1) -> dict:
    """
    월급여(세전, 월 단위)를 받아 연간 기준으로 근사 계산 후 월 실수령액을 산출.
    (상여금 등은 고려하지 않은 단순화 모델: 연봉 = 월급여 x 12)
    """
    총급여 = 월급여 * 12
    근로소득공제 = calc_근로소득공제(총급여)
    근로소득금액 = 총급여 - 근로소득공제

    사대보험_월 = calc_4대보험료(월급여)
    사대보험_연 = sum(사대보험_월.values()) * 12

    기본공제 = 인적공제수 * 1_500_000
    과세표준 = max(근로소득금액 - 기본공제 - 사대보험_연, 0)

    산출세액 = calc_산출세액(과세표준)
    근로소득세액공제 = calc_근로소득세액공제(산출세액, 총급여)
    결정세액 = max(산출세액 - 근로소득세액공제, 0)
    지방소득세 = 결정세액 * 0.10

    소득세_월 = 결정세액 / 12
    지방소득세_월 = 지방소득세 / 12
    공제총액_월 = 소득세_월 + 지방소득세_월 + sum(사대보험_월.values())
    실수령액_월 = 월급여 - 공제총액_월

    return {
        "월급여(세전)": round(월급여),
        "연간 총급여": round(총급여),
        "근로소득공제(연)": round(근로소득공제),
        "과세표준(연)": round(과세표준),
        "산출세액(연)": round(산출세액),
        "근로소득세액공제(연)": round(근로소득세액공제),
        "결정세액(연)": round(결정세액),
        "소득세(월)": round(소득세_월),
        "지방소득세(월)": round(지방소득세_월),
        **{f"{k}(월)": v for k, v in 사대보험_월.items()},
        "공제총액(월)": round(공제총액_월),
        "실수령액(월)": round(실수령액_월),
    }


# 빠른 검증: 월급 500만원
from pprint import pprint
pprint(calculate_take_home_pay(5_000_000))


{'건강보험(월)': 177250,
 '결정세액(연)': 4096133,
 '고용보험(월)': 45000,
 '공제총액(월)': 845683,
 '과세표준(연)': 40107552,
 '국민연금(월)': 225000,
 '근로소득공제(연)': 12750000,
 '근로소득세액공제(연)': 660000,
 '산출세액(연)': 4756133,
 '소득세(월)': 341344,
 '실수령액(월)': 4154317,
 '연간 총급여': 60000000,
 '월급여(세전)': 5000000,
 '장기요양보험(월)': 22954,
 '지방소득세(월)': 34134}


## 6. 근거 기반 설명 생성 (Gemini 무료 API)

계산 결과와 검색된 법조문을 함께 LLM에 넘겨 **"근거를 인용하는 자연어 설명"** 을 생성합니다.

### Gemini API 키 발급 (무료)
1. https://aistudio.google.com/apikey 접속 후 구글 계정으로 로그인
2. "Create API key" 클릭 (무료)
3. 이 노트북과 같은 폴더에 `.env` 파일을 만들고 아래처럼 저장:
   ```
   GEMINI_API_KEY=AIzaxxxxxxxxxxxxxxxxxxxx
   ```

키가 없으면 아래 셀은 자동으로 **LLM 없이 계산 결과 + 근거 조문만 텍스트로 출력**하는
폴백 모드로 동작합니다 (그래도 노트북은 끝까지 실행됩니다).


In [8]:
from dotenv import load_dotenv
from google import genai

load_dotenv()
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
GEMINI_MODEL = "gemini-3.5-flash-lite"  # 2026-07 기준 gemini-2.5-flash는 신규 사용자에게 차단됨

if GEMINI_API_KEY:
    gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    print("Gemini API 키 감지됨 — LLM 설명 생성 모드로 동작합니다.")
else:
    gemini_client = None
    print("GEMINI_API_KEY가 없습니다 — 폴백 모드(계산 결과 + 근거 조문만 출력)로 동작합니다.")


Gemini API 키 감지됨 — LLM 설명 생성 모드로 동작합니다.


In [9]:
## 계산 항목 -> 코퍼스의 정식 주제(topic, 즉 조제목) 라벨로 매핑
# (자유 서술형 질문이 아니라 '이 계산 항목의 근거가 필요하다'는 구조화된 조회이므로,
#  코퍼스의 정식 주제명을 그대로 질의로 사용합니다 — 위 retrieve()의 키워드 부스트와
#  결합하면 비슷한 문서끼리(예: 근로소득공제 vs 근로소득세액공제) 헷갈리지 않습니다.)
ITEM_QUERIES = {
    "근로소득공제": "근로소득공제",
    "세율": "세율",
    "근로소득세액공제": "근로소득세액공제",
    "국민연금": "국민연금료율",
    "건강보험": "건강보험료율",
    "장기요양보험": "장기요양보험료율",
    "고용보험": "고용보험료율",
    "지방소득세": "지방소득세",
}


def gather_evidence(top_k: int = 1) -> dict:
    """계산에 사용된 각 항목별로 가장 관련도 높은 청크를 1건씩 모은다."""
    evidence = {}
    for item, query in ITEM_QUERIES.items():
        doc, score = retrieve(query, top_k=top_k)[0]
        evidence[item] = {"doc": doc, "score": score}
    return evidence


def format_citation(doc: dict) -> str:
    citation = f"[{doc['law']} {doc['article']}] {doc['text']}"
    if doc.get("chunk_info"):
        citation += f" (일부 발췌 {doc['chunk_info']})"
    return citation


def build_llm_prompt(월급여: float, breakdown: dict, evidence: dict) -> str:
    근거_텍스트 = "\n".join(f"- {format_citation(v['doc'])}" for v in evidence.values())
    return f"""당신은 친절한 세무 설명 도우미입니다. 아래 계산 결과와 법조문 근거를 바탕으로,
일반인이 이해하기 쉽게 한국어로 설명해 주세요. 반드시 아래 제공된 근거 조문만 인용하고,
제공되지 않은 정보는 추측하지 마세요. 마지막엔 이 계산이 교육용 단순화 모델이라는 점을
짧게 안내하세요.

[사용자 정보]
월급(세전): {월급여:,.0f}원

[계산 결과]
{json.dumps(breakdown, ensure_ascii=False, indent=2)}

[근거 법조문]
{근거_텍스트}
"""


def generate_explanation(월급여: float) -> str:
    breakdown = calculate_take_home_pay(월급여)
    evidence = gather_evidence()

    if gemini_client is None:
        # 폴백: LLM 없이 계산 결과 + 근거만 정리해서 출력
        lines = [f"## 월급 {월급여:,.0f}원 실수령액 분석 (LLM 미사용 · 폴백 모드)\n"]
        for k, v in breakdown.items():
            lines.append(f"- {k}: {v:,.0f}" if isinstance(v, (int, float)) else f"- {k}: {v}")
        lines.append("\n### 근거 조문")
        for item, v in evidence.items():
            lines.append(f"- **{item}** — {format_citation(v['doc'])}")
        return "\n".join(lines)

    prompt = build_llm_prompt(월급여, breakdown, evidence)
    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
    )
    return response.text


print("함수 정의 완료")


함수 정의 완료


## 7. 데모 실행

월급 예시 3가지(300만원 / 500만원 / 1000만원)로 전체 파이프라인을 실행합니다.


In [10]:
for 월급 in [3_000_000, 5_000_000, 10_000_000]:
    print("=" * 70)
    print(generate_explanation(월급))
    print()


안녕하세요! 월급 계산 결과와 관련 세법을 바탕으로 알기 쉽게 설명해 드리겠습니다.

### 1. 연간 소득과 세금 계산 과정
* **연간 총급여 (36,000,000원)**: 월급여 300만 원을 1년간 받은 금액입니다.
* **근로소득공제 (연 10,650,000원)**: [소득세법 제47조]에 따라, 총급여가 1,500만원 초과 4,500만원 이하인 경우 "750만원 + (1,500만원을 초과하는 금액의 15%)"를 적용하여 세금을 매기기 전 일정 금액을 깎아줍니다.
* **과세표준 (연 20,464,536원)**: 총급여에서 근로소득공제를 반영한 뒤 산출된 세금 부과 기준 금액입니다.
* **산출세액 (연 1,809,680원)**: [소득세법 제55조]에 따라 과세표준(1,400만원 초과 5,000만원 이하 구간)에 "84만원 + (1,400만원을 초과하는 금액의 15퍼센트)" 세율을 적용하여 계산한 세금입니다.
* **근로소득세액공제 (연 716,000원)**: [소득세법 제59조]에 따라, 위에서 계산된 산출세액(130만원 초과)에 대해 "71만5천원 + (130만원을 초과하는 금액의 100분의 30)"을 적용하여 세액을 깎아줍니다. 총급여액이 3,300만원 이하이므로 공제 한도 규정(74만원) 내에 해당합니다.
* **결정세액 (연 1,093,680원)**: 산출세액에서 세액공제를 뺀, 최종적으로 내야 하는 연간 소득세입니다.
* **월 납부 세금**: 
  * **소득세(월 91,140원)**: 연간 결정세액을 12개월로 나눈 금액입니다.
  * **지방소득세(월 9,114원)**: [지방세법 제92조]에 따라 소득세액(결정세액)의 100분의 10에 해당하는 금액이 매월 함께 부과됩니다.

---

### 2. 4대 보험 및 공제 내역 (월 기준)
* **국민연금 (135,000원)**: [국민연금법 제88조]에 따라 근로자 본인은 기준소득월액의 1,000분의 45(4.5퍼센트)를 부담합니다.
* **건강보험 (106,350원)**: [국민건강보

안녕하세요! 친절한 세무 설명 도우미입니다. 세전 월급 5,000,000원을 기준으로 계산된 월 급여 및 공제 내역을 알기 쉽게 설명해 드리겠습니다.

---

### 1. 연간 총급여와 근로소득공제
* **연간 총급여 (60,000,000원):** 세전 월급 500만 원을 1년 동안 받은 금액(500만 원 × 12개월)입니다.
* **근로소득공제(연) (12,750,000원):** [소득세법 제47조]에 따라 총급여액이 4,500만 원을 초과하고 1억 원 이하인 경우, '1,200만원 + (4,500만원을 초과하는 금액의 5%)'를 적용하여 총급여에서 공제한 금액입니다. 이 공제를 거치면 세금을 매기는 기준인 과세표준을 구하기 위한 바탕이 마련됩니다.
* **과세표준(연) (40,107,552원):** 총급여에서 근로소득공제 등을 반영한 연간 과세표준 금액입니다.

### 2. 소득세 및 지방소득세 계산
* **산출세액(연) (4,756,133원):** [소득세법 제55조]에 따라 종합소득 과세표준(1,400만 원 초과 5,000만 원 이하 구간)에 해당하여 '84만원 + (1,400만원을 초과하는 금액의 15%)'의 세율을 적용하여 계산한 금액입니다.
* **근로소득세액공제(연) (660,000원):** [소득세법 제59조]에 따라 근로소득에 대한 산출세액에서 공제되는 금액이며, 총급여액이 3,300만 원 초과 7,000만 원 이하인 경우의 공제 한도 규정에 따라 적용되었습니다.
* **결정세액(연) (4,096,133원):** 산출세액에서 근로소득세액공제를 뺀 최종 연간 세금입니다.
* **소득세(월) (341,344원):** 연간 결정세액을 12개월로 나누어 매월 납부하는 소득세입니다.
* **지방소득세(월) (34,134원):** [지방세법 제92조]에 따라 소득세법에 따라 산출된 소득세액(결정세액)의 100분의 10에 해당하는 금액으로, 매월 소득세와 함께 징수됩니다.

### 3. 4대 보험 공제 내역
* **국민연금(월) (225,000원):**

안녕하세요! 세무 설명 도우미입니다. 사용자님의 세전 월급 10,000,000원을 바탕으로 한 급여와 세금 계산 결과를 근거 법조문을 토대로 알기 쉽게 설명해 드리겠습니다.

---

### 1. 연간 총급여 및 근로소득공제
* **연간 총급여**: 월급 10,000,000원에 12개월을 곱한 **120,000,000원**입니다.
* **근로소득공제(연)**: [소득세법 제47조]에 따라 총급여액이 1억원을 초과하는 경우, 공제액은 **1,475만원 + (1억원을 초과하는 금액의 100분의 2)**로 계산됩니다. 이에 따라 연간 **15,150,000원**이 공제됩니다.

### 2. 과세표준 및 산출세액
* **과세표준(연)**: 연간 총급여에서 근로소득공제 등을 반영한 결과 **94,279,104원**이 됩니다.
* **산출세액(연)**: [소득세법 제55조]에 따른 종합소득 세율에 따라 과세표준(8,800만원 초과 1억5천만원 이하 구간: 1,536만원 + 8,800만원 초과 금액의 35%)을 적용하여 **17,557,686원**이 산출됩니다.

### 3. 세액공제 및 결정세액 (소득세 및 지방소득세)
* **근로소득세액공제(연)**: [소득세법 제59조]에 따라 산출세액에서 공제되는 금액으로, 본 계산에서는 **500,000원**이 공제됩니다.
* **결정세액(연)**: 산출세액에서 세액공제를 뺀 연간 소득세는 **17,057,686원**입니다.
* **소득세(월)**: 연간 결정세액을 12개월로 나누어 매월 **1,421,474원**을 납부합니다.
* **지방소득세(월)**: [지방세법 제92조]에 따라 개인지방소득세는 소득세액(결정세액)의 100분의 10에 해당하므로, 매월 **142,147원**이 함께 징수됩니다.

### 4. 4대 보험 (월 공제액)
* **국민연금(월)**: [국민연금법 제88조]에 따라 근로자 본인 부담분은 기준소득월액의 1,000분의 45(4.5퍼센트)인 **265,500원**입니다.
* **건강보험(월)**: [국민건강

## 마무리

- **인제스천(Ingestion)**: DOCX 파일 하나를 `python-docx`로 직접 순회 — 문단·표를 원문
  순서 그대로 읽어 조문 단위 청크로 재구성 (표는 linearize해서 본문에 통합)
- **검색(Retrieval)**: 로컬 임베딩 모델 + 하이브리드(임베딩+키워드) 유사도 → 무료, 완전 오프라인 가능
- **계산(Rule)**: 순수 코드 → 무료, 결정론적, 환각 없음. 검증 과정에서 실제 계산 버그(제59조
  한도 비율)도 하나 발견해 고쳤습니다
- **생성(Generation)**: Gemini 무료 API → 근거를 인용한 자연어 설명만 담당 (토큰 사용량 최소화)

이 구조는 "LLM이 모든 걸 다 하게" 만들지 않고, **정확성이 중요한 부분(계산)은 코드에,
자연스러움이 중요한 부분(설명)은 LLM에** 맡기는 전형적인 RAG + 툴 사용 패턴입니다.

원본 데이터 검증 과정(JSONL 원문 조사 → 표 이미지 직접 대조 → 하나의 DOCX로 정리)은
`preprocessing.ipynb`에 남아 있습니다. 실제 서비스로 확장한다면: (1) 이 DOCX와
무관한 나머지 조문(양도소득세율 등)도 동일한 방식으로 검증, (2) 최신 4대보험 요율 자동
업데이트, (3) 근로소득 간이세액표 정확 반영, (4) 부양가족·비과세소득 등 실제 변수 반영이
필요합니다.
